We make use of QC Design's state of the art QEC simulation package `plaquette` to represent the repetition code graphically. We start by first importing the necessary components:

In [27]:
import numpy as np
import plaquette
from plaquette.codes import LatticeCode
from plaquette import visualizer

Now we need to define a quantum error correction code. In the case of the quantum repetition code, the parameter `size` stands for the distance of the code with respect to logical $\bar{X}$ errors. For now, the parameter `n_rounds` is left as $1$.

In [28]:
repetition_code = LatticeCode.make_repetition(size=3, n_rounds=1)

`plaquette` has a built-in visualizer, where the red circles represent the data-qubits, the blue crosses represent the ancillas responsible for the stabilizer measurements, the blue lines represent $CNOT$ gates, the green dashed lines represent $CZ$ gates, and the yellow stars represent the logical operators. This visualizer is an interactive tool: you can hover your mouse cursor on top of the various elements to display some more information.

In [30]:
vis = visualizer.LatticeVisualizer(repetition_code)
vis.draw_lattice(height=300)

<div style="color: var(--nextui-colors-text); text-transform: none; background-color: white; margin: 20px 0px; padding: 20px; border-left: 3px solid;">
If we run this on a quantum computer, we would need to express this idea as a circuit like any other quantum algorithm. The left part encodes the qubit with the repetition code. Then (after a potential bit-flip noise) two additional ancillary qubits are used to perform the stabilizer measurements.
![circuit](/img/learn/qec/repetition-code-circuit.svg)

</div>

Now, let's use `plaquette` to generate errors on the code, perform the stabilizer measurements and obtain a syndrome. For this, we will need some additional modules:

In [31]:
from plaquette.errors import QubitErrorsDict
from plaquette.device import Device, MeasurementSample
from plaquette.circuit.generator import generate_qec_circuit

<div style="color: var(--nextui-colors-text); text-transform: none; background-color: white; margin: 20px 0px; padding: 20px; border-left: 3px solid;">
This demonstration uses the open source version of plaquette, which is not available in general anymore., An archived version of the docs can be found [here](https://web.archive.org/web/20231220123913/https://docs.plaquette.design/en/stable/).</div>


As we will need this more often, we will define a function `get_syndrome_from_deterministic_error` to obtain the syndrome of a code when we place specific errors on the qubits. This function has three parameters: 

- `code`: an instance of `plaquette`'s error correction codes, `LatticeCode`. 
- `qubit_errors`: a dictionary containing the qubits and the errors that you want to apply on those qubits. 
- `logical_ops`: a string necessary to create the quantum circuit used in simulations. At this moment, it is not necessary for you to worry about this last requirement.

In [32]:
def get_syndrome_from_deterministic_error(code, qubit_errors: dict[int: list[str]], logical_ops="X"):
    qed: QubitErrorsDict = {
        "pauli": {i: {error: 1 for error in qubit_errors.get(i)} for i in qubit_errors.keys()}
    }
    circuit = generate_qec_circuit(code, qed, {}, logical_ops)
    device = Device("clifford")
    device.run(circuit)
    raw_results, erasure = device.get_sample()
    sample = MeasurementSample.from_code_and_raw_results(code, raw_results, erasure)
    return sample.syndrome[0]

<div style="color: var(--nextui-colors-text); text-transform: none; background-color: white; margin: 20px 0px; padding: 20px; border-left: 3px solid;">
Check out `plaquette`'s [Quickstart guide](https://docs.plaquette.design/quickstart.html) to learn more about everything used in the function. A short summary: first, we create a dictionary `qed` that contains the error probabilities for each type of error on each qubit. For now, this probability is set to 1 because we want to apply these errors deterministically. Then, a `circuit` is generated. This is the circuit where the quantum code is initialized, the errors appear, and the stabilizer measurements are carried out. Then, a new "device" is created to run the `circuit` we generated, and it is used to obtain a sample of measurements and erasures (the latter are ingored for the remainder of this tutorial). The raw results are processed into a syndrome, and the syndrome is returned.</div>

Use the code block below to represent graphically the syndrome of a given error. Feel free to change the `qubit_errors` parameter and re-run the code block to see the syndrome of your new error. The default error is an $X$ error on the first qubit of the chain. You can add as many errors as you want, and observe the syndromes obtained in each case. To do this, change the parameter <span class="title-ref">qubit_errors</span> provided to the <span class="title-ref">get_syndrome_from_deterministic_error</span> method. We suggest starting with the the following errors:

-   `{0: ["x"]}`
-   `{1: ["x"]}`
-   `{2: ["x"]}`

In [26]:
syndrome = get_syndrome_from_deterministic_error(code = repetition_code, qubit_errors = {0:["x"]})
vis.draw_latticedata(height=300, syndrome=syndrome)

If you have tested the three suggested errors and observed carefully the syndromes obtained with each one of them, you will be starting to understand how are the individual $X$ error reflected in the syndrome. Now, try to input the following errors, which contain errors on multiple individual qubits, and compare this syndromes with the ones obtained above:

-   `{1: ["x"], 2: ["x"]}`
-   `{0: ["x"], 2: ["x"]}`
-   `{0: ["x"], 1: ["x"]}`

From these new examples, you might have learned two things:

-   First of all, the measurement result of an ancilla does not tell you exactly if an error appeared in its neighboring qubits, it only     tells you that an error anti-commutes with the stabilizer measured     with that ancilla. If we only had a single error `{0: ["x"]}`, we     see that the first stabilizer measurement is toggled. If we only     have the error `{1: ["x"]}`, we see that both stabilizer     measurements are toggled (this error anti-commutes with two     stabilizers). But when we apply the composed error     `{0: ["x"], 1:["x"]}`, only one stabilizer measurement is toggled.     This is because the first stabilizer, $Z_0Z_1$ anti-commutes with $X_0$, but it also anti-commutes with $X_1$. The anti-commutation of    the two is cancelled out, and thus $Z_0Z_1$ commutes with $X_0X_1$.    A stabilizer measurement is toggled only when there is an odd amount    of operators in the error that anti-commute with the stabilizer.
-   Second, you may have noticed that the syndromes obtained by these    errors are the same as the syndromes obtained with the previous set    of errors. This tells us that a syndrome is degenerate, i.e., different errors can give the same syndrome. Good that two-qubit errors are less probable than one-qubit errors.

Try to also use the following error `{0: ["x"], 1: ["x"], 2: ["x"]}`. As you might notice, this error toggles no stabilizer! This is because this error is actually the logical error $\bar{X}$, which is not detectable.

Maybe you can already guess what happens when applying $Z$ errors, such as:

-   `{0: ["z"]}`
-   `{1: ["z"]}`
-   `{2: ["z"]}`
-   `{0: ["z"], 1: ["z"]}`
-   `{1: ["z"], 2: ["z"]}`
-   `{0: ["z"], 2: ["z"]}`

As you can see, the measurement is always the same, we are not detecting any errors at all! This is because the stabilizers of the code are all products of $Z$ operators, and then $Z$ errors commute with all of them, and these types of errors will be forever undetected. This means we need to improve our quantum error correction code, by adding *something* capable of detecting both types of errors.